# Chapter 10: Files and Exceptions

> En este capítulo se aprende a leer y escribir archivos, a manejar errores con excepciones y a guardar datos con el módulo `json`.

## Summary

Con el módulo **`pathlib`** se crea un objeto `Path` que apunta a un archivo; `read_text()` lee todo su contenido como un string, `splitlines()` lo separa en líneas y `write_text()` escribe un string en un archivo (creándolo si no existe y **borrando** lo que tenía si ya existía). Las rutas pueden ser **relativas** (desde la carpeta del programa) o **absolutas** (desde la raíz del sistema).
Una **excepción** es un objeto especial que Python crea cuando ocurre un error. Con un bloque **`try-except`** se indica qué hacer si ocurre una excepción en particular (`ZeroDivisionError`, `FileNotFoundError`, `ValueError`...), y con **`else`** se pone el código que depende de que el `try` haya funcionado. Con `pass` un programa puede fallar en silencio.
El módulo **`json`** guarda estructuras de datos de Python en un archivo (`json.dumps()`) y las recupera (`json.loads()`), de modo que los datos del usuario no se pierdan cuando el programa termina. `path.exists()` permite saber si un archivo existe.
Se vuelve a hacer **refactoring**: dividir el código en funciones con un propósito claro (`get_stored_username()`, `get_new_username()`, `greet_user()`).

> Para ejecutar las celdas de lectura de archivos hay que tener en la misma carpeta del notebook los archivos `pi_digits.txt`, `pi_million_digits.txt`, `alice.txt`, `moby_dick.txt` y `little_women.txt`. Están en los recursos del libro (en `pcc_3e-main/chapter_10`). Las celdas que usan `input()` no tienen salida guardada.

## Cheat Sheet

### Funciones y métodos

| Quiero... | Sintaxis | Ejemplo | Resultado / nota |
|---|---|---|---|
| Importar `Path` | `from pathlib import Path` | | `pathlib` hace más fácil trabajar con archivos y carpetas en cualquier sistema operativo |
| Apuntar a un archivo | `Path('ruta')` | `path = Path('pi_digits.txt')` | solo el nombre si está en la carpeta del programa |
| Leer todo el archivo | `path.read_text()` | `contents = path.read_text()` | un solo string (con una línea en blanco al final) |
| Leer con una codificación | `path.read_text(encoding='utf-8')` | `path.read_text(encoding='utf-8')` | necesario si la codificación del sistema no coincide con la del archivo |
| Quitar el espacio final | `.rstrip()` | `path.read_text().rstrip()` | encadenar métodos = *method chaining* |
| Separar un string en líneas | `.splitlines()` | `contents.splitlines()` | lista con una línea por elemento |
| Recorrer las líneas | `for line in contents.splitlines():` | | no hace falta una variable temporal |
| Quitar espacios a la izquierda | `.lstrip()` | `line.lstrip()` | |
| Convertir texto leído en número | `int()` / `float()` | | todo lo que se lee de un archivo es un string |
| Escribir en un archivo | `path.write_text(texto)` | `path.write_text("I love programming.")` | crea el archivo si no existe; **borra** el contenido si ya existía |
| Convertir un número para escribirlo | `str()` | | un archivo de texto solo guarda strings |
| Saber si un archivo existe | `path.exists()` | `if path.exists():` | `True` / `False` |
| Separar un texto en palabras | `.split()` | `contents.split()` | lista de palabras (separa por espacios en blanco) |
| Reemplazar texto | `.replace(a, b)` | `"I really like dogs.".replace('dog', 'cat')` | `'I really like cats.'` |
| Contar apariciones | `.count(x)` | `"Row, row, row your boat".lower().count('row')` | `3` |
| Importar json | `import json` | | |
| Pasar un dato a texto JSON | `json.dumps(dato)` | `json.dumps([2, 3, 5, 7, 11, 13])` | devuelve un string |
| Pasar texto JSON a dato | `json.loads(texto)` | `json.loads(contents)` | devuelve un objeto de Python (lista, diccionario, string...) |
| No hacer nada en un bloque | `pass` | `except FileNotFoundError: pass` | también sirve como marcador |

### Estructura try-except

| Parte | Qué hace |
|---|---|
| `try:` | contiene **solo** el código que podría causar la excepción |
| `except TipoDeError:` | qué hacer si ocurre esa excepción (el tipo se ve en la última línea del traceback) |
| `else:` | código que depende de que el `try` haya funcionado |
| `pass` | para fallar en silencio |

### Rutas

| Tipo | Qué es | Ejemplo |
|---|---|---|
| Relativa | desde la carpeta donde está el programa | `Path('text_files/filename.txt')` |
| Absoluta | desde la raíz del sistema | `Path('/home/eric/data_files/text_files/filename.txt')` |

En el código se usan siempre barras `/`, incluso en Windows (`pathlib` usa la correcta en cada sistema).

### Errores del capítulo

| Error | Mensaje | Causa | Solución |
|---|---|---|---|
| `ZeroDivisionError` | `division by zero` | dividir un número por cero (`5/0`) | `try-except ZeroDivisionError` |
| `FileNotFoundError` | `[Errno 2] No such file or directory: 'alice.txt'` | el archivo no está en la ruta indicada, o el nombre está mal escrito | `try-except FileNotFoundError` o revisar la ruta |
| `ValueError` | (al convertir texto a número con `int()`) | el usuario ingresó texto en vez de un número | `try-except ValueError` |
| Archivo borrado sin querer | (no hay mensaje) | `write_text()` sobre un archivo existente | revisar con `path.exists()` antes de escribir |

## Reading from a File

Hay una cantidad increíble de datos disponibles en archivos de texto: datos del clima, del tráfico, socioeconómicos, obras literarias, etc. Leer de un archivo es especialmente útil en el análisis de datos, pero sirve en cualquier situación en que se quiera analizar o modificar la información guardada en un archivo.
El primer paso para trabajar con la información de un archivo de texto es **leerlo en memoria**. Después se puede trabajar con todo el contenido a la vez o línea por línea.

### Reading the Contents of a File

Se parte con un archivo que contiene el número pi con 30 decimales, 10 por línea (`pi_digits.txt`):

```
3.1415926535
  8979323846
  2643383279
```

Programa que abre el archivo, lo lee y muestra su contenido:

In [1]:
from pathlib import Path

path = Path('pi_digits.txt')
contents = path.read_text()
print(contents)

3.1415926535
  8979323846
  2643383279



- Para trabajar con el contenido de un archivo hay que decirle a Python la **ruta** (*path*): la ubicación exacta de un archivo o carpeta en un sistema.
- `pathlib` es un módulo que facilita el trabajo con archivos y directorios sin importar el sistema operativo. Un módulo que da una funcionalidad específica como ésta se suele llamar **biblioteca** (*library*).
- Se importa la clase `Path` de `pathlib`. Con un objeto `Path` que apunta a un archivo se puede, por ejemplo, revisar que el archivo exista, leer su contenido o escribir datos nuevos. Aquí se crea uno para `pi_digits.txt` y se asigna a `path`. Como el archivo está en la misma carpeta que el programa, basta con el nombre.
- `read_text()` lee **todo** el contenido del archivo y lo devuelve como un solo string, que se asigna a `contents`.

> VS Code busca los archivos en la última carpeta que se abrió. Hay que abrir primero la carpeta donde se guardan los archivos del capítulo.

La única diferencia con el archivo original es la línea en blanco extra al final, que aparece porque `read_text()` devuelve un string vacío al llegar al final del archivo. Se puede quitar con `rstrip()`:

In [2]:
from pathlib import Path

path = Path('pi_digits.txt')
contents = path.read_text()
contents = contents.rstrip()
print(contents)

3.1415926535
  8979323846
  2643383279


Se puede aplicar `rstrip()` apenas se llama a `read_text()`:

```python
contents = path.read_text().rstrip()
```

Esta línea le dice a Python que llame al método `read_text()` y luego aplique `rstrip()` al string que devuelve. Este enfoque se llama **method chaining** (encadenamiento de métodos) y se usa mucho en programación.

### Relative and Absolute File Paths

Cuando a `Path` se le pasa solo un nombre de archivo, como `pi_digits.txt`, Python busca en el directorio donde está el programa que se está ejecutando. A veces el archivo no está ahí. Por ejemplo, los programas pueden estar en una carpeta `python_work` y los archivos de texto en `text_files`, dentro de `python_work`. Aunque `text_files` está dentro de `python_work`, pasar solo el nombre del archivo no funciona, porque Python solo busca en `python_work`. Hay que indicar la ruta correcta.

Hay dos formas principales de especificar rutas:
- Una **ruta relativa** le dice a Python que busque un lugar dado **relativo al directorio** donde está el programa. Como `text_files` está en `python_work`, se construye una ruta que parte por `text_files` y termina con el nombre del archivo:

```python
path = Path('text_files/filename.txt')
```

- Una **ruta absoluta** le dice a Python exactamente dónde está el archivo en el computador, sin importar dónde esté el programa. Se usa si una ruta relativa no funciona. Normalmente son más largas, porque parten en la carpeta raíz del sistema:

```python
path = Path('/home/eric/data_files/text_files/filename.txt')
```

Con rutas absolutas se pueden leer archivos de cualquier parte del sistema. Por ahora lo más fácil es guardar los archivos en la misma carpeta que los programas, o en una carpeta como `text_files` dentro de ella.

> Windows muestra las rutas con barra invertida (`\`) en vez de barra (`/`), pero en el código hay que usar siempre barras `/`, incluso en Windows. La biblioteca `pathlib` usa automáticamente la representación correcta.

### Accessing a File's Lines

Al trabajar con un archivo, a menudo se quiere examinar cada línea: para buscar cierta información o para modificar el texto. Por ejemplo, en un archivo de datos del clima se podría trabajar con cualquier línea que incluya la palabra *sunny*, o en un reporte de noticias buscar las líneas con la etiqueta `<headline>` y reescribirlas con un formato específico.
Con el método `splitlines()` se convierte un string largo en un conjunto de líneas, y con un `for` se examina cada una:

In [3]:
from pathlib import Path

path = Path('pi_digits.txt')
contents = path.read_text()

lines = contents.splitlines()
for line in lines:
    print(line)

3.1415926535
  8979323846
  2643383279


Primero se lee todo el contenido del archivo. Si se va a trabajar con las líneas individuales, no hace falta quitar los espacios en blanco al leer. `splitlines()` devuelve una **lista** con todas las líneas del archivo, que se asigna a `lines`, y luego un `for` recorre y muestra cada una. Como no se modificó nada, la salida coincide con el archivo original.

### Working with a File's Contents

Una vez leído el contenido de un archivo en memoria, se puede hacer lo que se quiera con esos datos. Se intenta armar un solo string con todos los dígitos de pi, sin espacios:

In [4]:
from pathlib import Path

path = Path('pi_digits.txt')
contents = path.read_text()

lines = contents.splitlines()
pi_string = ''
for line in lines:
    pi_string += line

print(pi_string)
print(len(pi_string))

3.1415926535  8979323846  2643383279
36


`pi_string` contiene los espacios en blanco que había a la izquierda de los dígitos de cada línea. Se eliminan con `lstrip()` en cada línea:

In [5]:
from pathlib import Path

path = Path('pi_digits.txt')
contents = path.read_text()

lines = contents.splitlines()
pi_string = ''
for line in lines:
    pi_string += line.lstrip()

print(pi_string)
print(len(pi_string))

3.141592653589793238462643383279
32


Ahora el string tiene pi con 30 decimales. Mide 32 caracteres porque también incluye el `3` inicial y el punto decimal.

> Cuando Python lee un archivo de texto, **interpreta todo el texto como un string**. Si se lee un número y se quiere usar en un contexto numérico, hay que convertirlo con `int()` o `float()`.

### Large Files: One Million Digits

El código sirve igual para archivos mucho más grandes. Si el archivo tiene pi con 1.000.000 de decimales, no hay que cambiar el programa, salvo pasarle otro archivo. Se imprimen solo los primeros 50 decimales, para no ver un millón de dígitos pasar por la pantalla:

In [6]:
from pathlib import Path

path = Path('pi_million_digits.txt')
contents = path.read_text()

lines = contents.splitlines()
pi_string = ''
for line in lines:
    pi_string += line.lstrip()

print(f"{pi_string[:52]}...")
print(len(pi_string))

3.14159265358979323846264338327950288419716939937510...
1000002


Python no tiene un límite inherente a la cantidad de datos con los que se puede trabajar: se puede trabajar con tantos datos como la memoria del sistema soporte.

### Is Your Birthday Contained in Pi?

Con el programa anterior se puede averiguar si el cumpleaños de alguien aparece en el primer millón de dígitos de pi. Se expresa la fecha como un string de dígitos y se revisa si aparece en `pi_string`:

In [ ]:
from pathlib import Path

path = Path('pi_million_digits.txt')
contents = path.read_text()

lines = contents.splitlines()
pi_string = ''
for line in lines:
    pi_string += line.strip()

birthday = input("Enter your birthday, in the form mmddyy: ")
if birthday in pi_string:
    print("Your birthday appears in the first million digits of pi!")
else:
    print("Your birthday does not appear in the first million digits of pi.")

Una vez leído un archivo, se puede analizar su contenido de casi cualquier forma imaginable.

### Comparison: Reading from a File

| | `read_text()` | `read_text().rstrip()` | `read_text().splitlines()` |
|---|---|---|---|
| Devuelve | un string con todo el archivo | el mismo string, sin el espacio final | una lista con una línea por elemento |
| Con `pi_digits.txt` | `'3.1415926535\n  8979323846\n  2643383279\n'` | `'3.1415926535\n  8979323846\n  2643383279'` | `['3.1415926535', '  8979323846', '  2643383279']` |
| Para qué sirve | ver o procesar todo el texto | mostrarlo sin la línea en blanco | examinar cada línea con un `for` |

| | `.rstrip()` | `.lstrip()` | `.strip()` |
|---|---|---|---|
| Quita espacios de | la derecha | la izquierda | ambos lados |
| Se usó en este capítulo | al leer todo el archivo | en cada línea de `pi_string` | en el programa del cumpleaños |

| | Ruta relativa | Ruta absoluta |
|---|---|---|
| Parte desde | la carpeta del programa | la raíz del sistema |
| Ejemplo | `Path('text_files/filename.txt')` | `Path('/home/eric/data_files/text_files/filename.txt')` |
| Cuándo usarla | los archivos están cerca del programa | el archivo está en cualquier otro lugar |

### Try it Yourself

> 10-1. Learning Python:
Open a blank file in your text editor and write a few lines summarizing what you’ve learned about Python so far. Start each line with the phrase In Python you can. . . . Save the file as learning_python.txt in the same directory as your exercises from this chapter. Write a program that reads the file and prints what you wrote two times: print the contents once by reading in the entire file, and once by storing the lines in a list and then looping over each line.

> 10-2. Learning C:
You can use the replace() method to replace any word in a string with a different word. Here’s a quick example showing how to replace 'dog' with 'cat' in a sentence:

In [7]:
message = "I really like dogs."
print(message.replace('dog', 'cat'))

I really like cats.


Read in each line from the file you just created, learning_python.txt, and replace the word Python with the name of another language, such as C. Print each modified line to the screen.

> 10-3. Simpler Code:
The program file file_reader.py in this section uses a temporary variable, lines, to show how splitlines() works. You can skip the temporary variable and loop directly over the list that splitlines() returns:

```python
for line in contents.splitlines():
```

Remove the temporary variable from each of the programs in this section, to make them more concise.

In [ ]:
# 10-1

In [ ]:
# 10-2

In [ ]:
# 10-3

## Writing to a File

Una de las formas más simples de guardar datos es escribirlos en un archivo. Así la salida sigue disponible después de cerrar la terminal con la salida del programa, se puede examinar cuando el programa termina y se puede compartir. También se puede escribir un programa que lea el texto de vuelta en memoria y trabaje con él después.

### Writing a Single Line

Cuando ya hay una ruta definida, se puede escribir en el archivo con el método `write_text()`. Por ejemplo, para guardar un mensaje simple en un archivo en vez de imprimirlo:

In [8]:
from pathlib import Path

path = Path('programming.txt')
path.write_text("I love programming.")

`write_text()` recibe un solo argumento: el string que se quiere escribir. El programa no muestra nada en la terminal, pero si se abre `programming.txt` se ve una línea:

```
I love programming.
```

El archivo se comporta como cualquier otro del computador: se puede abrir, escribir en él, copiarlo, etc.

> Python solo puede escribir **strings** en un archivo de texto. Si se quieren guardar datos numéricos, hay que convertirlos primero con `str()`.

### Writing Multiple Lines

`write_text()` hace algunas cosas por detrás: si el archivo al que apunta `path` no existe, lo crea; y después de escribir, se asegura de cerrar el archivo correctamente (los archivos que no se cierran bien pueden provocar datos faltantes o corruptos).
Para escribir más de una línea hay que armar un string con todo el contenido del archivo y llamar a `write_text()` con ese string:

In [9]:
from pathlib import Path

contents = "I love programming.\n"
contents += "I love creating new games.\n"
contents += "I also love working with data.\n"

path = Path('programming.txt')
path.write_text(contents)

La variable `contents` guarda todo el contenido del archivo. Con `+=` se agrega al string tantas veces como haga falta, y se incluyen saltos de línea (`\n`) al final de cada línea para que cada frase quede en su propia línea. También se pueden usar espacios, tabulaciones y líneas en blanco, igual que en la salida de la terminal; no hay límite de largo, y así se crean muchos documentos generados por computador.

```
I love programming.
I love creating new games.
I also love working with data.
```

> Hay que tener cuidado al llamar a `write_text()`: si el archivo ya existe, **borra el contenido actual** y escribe el nuevo. Más adelante se verá cómo comprobar si un archivo existe con `pathlib`.

### Comparison: Writing to a File

| | `read_text()` | `write_text(texto)` |
|---|---|---|
| Qué hace | lee el contenido de un archivo | escribe un string en un archivo |
| Devuelve | un string | (nada que se use) |
| Si el archivo no existe | error (`FileNotFoundError`) | **lo crea** |
| Si el archivo existe | lo lee | **borra** el contenido anterior y escribe el nuevo |

| | Una línea | Varias líneas |
|---|---|---|
| Cómo se arma el texto | un solo string | un string armado con `+=` y `\n` al final de cada línea |
| Llamadas a `write_text()` | una | una, con todo el contenido |
| Resultado en el archivo | `I love programming.` | tres líneas, una por frase |

### Try it Yourself

> 10-4. Guest:
Write a program that prompts the user for their name. When they respond, write their name to a file called guest.txt.

> 10-5. Guest Book:
Write a while loop that prompts users for their name. Collect all the names that are entered, and then write these names to a file called guest_book.txt. Make sure each entry appears on a new line in the file.

In [ ]:
# 10-4

In [ ]:
# 10-5

## Exceptions

Python usa objetos especiales llamados **excepciones** para manejar los errores que aparecen mientras se ejecuta un programa. Cuando ocurre un error que hace que Python no sepa qué hacer, crea un objeto de excepción. Si hay código que maneja la excepción, el programa continúa; si no, el programa se detiene y muestra un **traceback**, que incluye un reporte de la excepción.
Las excepciones se manejan con bloques **`try-except`**: le piden a Python que haga algo y le dicen qué hacer si se produce una excepción. Así el programa sigue corriendo aunque algo salga mal, y en vez de tracebacks (que pueden ser confusos para los usuarios) se muestran mensajes amables escritos por uno.

### Handling the ZeroDivisionError Exception

Un error simple que hace que Python produzca una excepción: pedirle dividir por cero.

```python
print(5/0)
```

```
Traceback (most recent call last):
  File "division_calculator.py", line 1, in <module>
    print(5/0)
          ~^~
ZeroDivisionError: division by zero
```

`ZeroDivisionError` es un objeto de excepción. Python crea este tipo de objeto cuando no puede hacer lo que se le pide; entonces detiene el programa e informa qué tipo de excepción se produjo. Con esa información se puede decir qué hacer cuando ocurra ese tipo de excepción.

### Using try-except Blocks

Cuando se cree que puede ocurrir un error, se escribe un bloque `try-except` para manejar la excepción que se podría producir. Se le dice a Python que intente ejecutar cierto código, y qué hacer si el código produce un tipo de excepción en particular:

In [10]:
try:
    print(5/0)
except ZeroDivisionError:
    print("You can't divide by zero!")

You can't divide by zero!


La línea que causó el error (`print(5/0)`) va dentro del bloque `try`. Si el código del `try` funciona, Python se salta el bloque `except`. Si causa un error, Python busca un bloque `except` cuyo error coincida con el que se produjo, y ejecuta su código. Así el usuario ve un mensaje amable en vez de un traceback.
Si hubiera más código después del bloque `try-except`, el programa continuaría ejecutándose, porque se le dijo a Python cómo manejar el error.

### Using Exceptions to Prevent Crashes

Manejar los errores correctamente es especialmente importante cuando el programa tiene más trabajo que hacer después del error. Pasa a menudo en programas que piden información al usuario: si el programa responde de forma apropiada a una entrada inválida, puede pedir otra entrada válida en vez de caerse.
Una calculadora simple que solo divide:

In [ ]:
print("Give me two numbers, and I'll divide them.")
print("Enter 'q' to quit.")

while True:
    first_number = input("\nFirst number: ")
    if first_number == 'q':
        break
    second_number = input("Second number: ")
    if second_number == 'q':
        break
    answer = int(first_number) / int(second_number)
    print(answer)

Este programa no maneja errores: si se le pide dividir por cero, se cae con un `ZeroDivisionError`.
Es malo que el programa se caiga, pero tampoco es bueno que los usuarios vean tracebacks: confunden a los usuarios no técnicos, y en un contexto malicioso un atacante aprende más de lo que se quisiera (el nombre del archivo del programa, y una parte del código que no funciona bien); un atacante hábil puede usar esa información para decidir qué tipo de ataque usar.

### The else Block

El programa se hace más resistente a errores envolviendo la línea que podría producirlos en un bloque `try-except`. Se agrega un bloque **`else`**: todo código que **depende** de que el `try` funcione bien va en el `else`.

In [ ]:
print("Give me two numbers, and I'll divide them.")
print("Enter 'q' to quit.")

while True:
    first_number = input("\nFirst number: ")
    if first_number == 'q':
        break
    second_number = input("Second number: ")
    if second_number == 'q':
        break
    try:
        answer = int(first_number) / int(second_number)
    except ZeroDivisionError:
        print("You can't divide by 0!")
    else:
        print(answer)

- Se le pide a Python que intente la división en un bloque `try`, que incluye **solo** el código que podría causar un error.
- El bloque `except ZeroDivisionError` dice cómo responder cuando se produce esa excepción: se muestra un mensaje amable que dice cómo evitar el error. El programa sigue corriendo y el usuario nunca ve un traceback.
- Si la división funciona, se usa el bloque `else` para mostrar el resultado.

Con esto el programa responde a una división por cero (`You can't divide by 0!`) y sigue con la siguiente pregunta; con `5` y `2` muestra `2.5`; y termina cuando se ingresa `q`.

Solo el código que podría provocar una excepción va en el `try`. El código que debe ejecutarse únicamente si el `try` tuvo éxito va en el `else`. El `except` dice qué hacer si surge cierta excepción. Si se anticipan las fuentes probables de error, se escriben programas robustos que siguen corriendo ante datos inválidos o recursos que faltan, resistentes a errores inocentes de los usuarios y a ataques maliciosos.

### Handling the FileNotFoundError Exception

Un problema común al trabajar con archivos es que **falten**: el archivo puede estar en otra ubicación, el nombre puede estar mal escrito, o puede que ni siquiera exista. Todo eso se maneja con un bloque `try-except`.
Un programa que intenta leer *Alice in Wonderland* (`alice.txt`) cuando el archivo no está en la misma carpeta:

```python
from pathlib import Path

path = Path('alice.txt')
contents = path.read_text(encoding='utf-8')
```

El argumento `encoding` hace falta cuando la codificación por defecto del sistema no coincide con la del archivo que se lee; es más probable cuando se lee un archivo que no se creó en el propio sistema. Python no puede leer un archivo que no existe, y produce una excepción:

```
Traceback (most recent call last):
  File "alice.py", line 4, in <module>
    contents = path.read_text(encoding='utf-8')
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/.../pathlib.py", line 1056, in read_text
    ...
FileNotFoundError: [Errno 2] No such file or directory: 'alice.txt'
```

Es un traceback más largo. Casi siempre conviene **empezar por el final**: en la última línea se ve que se produjo un `FileNotFoundError`, que es el tipo de excepción que hay que usar en el `except`. Al inicio se ve en qué línea del archivo ocurrió el error y qué código lo causó. El resto muestra código de las bibliotecas involucradas, que normalmente no hace falta leer.
El bloque `try` empieza con la línea que el traceback identificó como problemática:

In [11]:
from pathlib import Path

path = Path('alice.txt')
try:
    contents = path.read_text(encoding='utf-8')
except FileNotFoundError:
    print(f"Sorry, the file {path} does not exist.")

Sorry, the file alice.txt does not exist.


> Esta celda se ejecutó para mostrar el caso del archivo que falta. Si `alice.txt` está en tu carpeta, no mostrará el mensaje y no imprimirá nada.

El programa no tiene nada más que hacer si el archivo no existe, así que esa es toda la salida.

### Analyzing Text

Se pueden analizar archivos de texto que contienen libros completos. Muchas obras clásicas están disponibles como texto simple, porque son de dominio público. Los textos de esta sección vienen de Project Gutenberg (https://gutenberg.org), una colección de obras literarias de dominio público.
Se trae el texto de *Alice in Wonderland* y se cuenta el número de palabras con el método `split()`, que por defecto separa un string donde encuentre cualquier espacio en blanco:

In [12]:
from pathlib import Path

path = Path('alice.txt')
try:
    contents = path.read_text(encoding='utf-8')
except FileNotFoundError:
    print(f"Sorry, the file {path} does not exist.")
else:
    # Count the approximate number of words in the file:
    words = contents.split()
    num_words = len(words)
    print(f"The file {path} has about {num_words} words.")

The file alice.txt has about 29594 words.


El string `contents` contiene todo el texto como un solo string largo; `split()` produce una lista con todas las palabras del libro, y `len()` de esa lista da una buena aproximación de la cantidad de palabras. Este código va en el `else` porque solo funciona si el `try` tuvo éxito.
La cuenta es un poco alta porque el archivo trae información extra del editor, pero es una buena aproximación del largo.

### Working with Multiple Files

Se agregan más libros, pero antes se mueve la mayor parte del programa a una función `count_words()`, para facilitar el análisis de varios libros:

In [13]:
from pathlib import Path

def count_words(path):
    """Count the approximate number of words in a file."""
    try:
        contents = path.read_text(encoding='utf-8')
    except FileNotFoundError:
        print(f"Sorry, the file {path} does not exist.")
    else:
        # Count the approximate number of words in the file:
        words = contents.split()
        num_words = len(words)
        print(f"The file {path} has about {num_words} words.")

path = Path('alice.txt')
count_words(path)

The file alice.txt has about 29594 words.


Casi todo el código quedó igual: solo se indentó y se movió al cuerpo de `count_words()`. Es buena costumbre mantener los comentarios al día al modificar un programa, así que el comentario pasó a ser un **docstring**.

Ahora un `for` cuenta las palabras de cualquier texto: se guardan los nombres de los archivos en una lista y se llama a `count_words()` para cada uno. Se usan *Alice in Wonderland*, *Siddhartha*, *Moby Dick* y *Little Women*. Se dejó `siddhartha.txt` afuera a propósito, para ver cómo maneja el programa un archivo que falta:

In [14]:
from pathlib import Path

def count_words(path):
    """Count the approximate number of words in a file."""
    try:
        contents = path.read_text(encoding='utf-8')
    except FileNotFoundError:
        print(f"Sorry, the file {path} does not exist.")
    else:
        # Count the approximate number of words in the file:
        words = contents.split()
        num_words = len(words)
        print(f"The file {path} has about {num_words} words.")

filenames = ['alice.txt', 'siddhartha.txt', 'moby_dick.txt',
        'little_women.txt']
for filename in filenames:
    path = Path(filename)
    count_words(path)

The file alice.txt has about 29594 words.
Sorry, the file siddhartha.txt does not exist.
The file moby_dick.txt has about 215864 words.
The file little_women.txt has about 189142 words.


Los nombres de los archivos se guardan como strings simples, y cada uno se convierte en un objeto `Path` antes de llamar a `count_words()`. El archivo `siddhartha.txt` que falta no afecta al resto de la ejecución.
Usar `try-except` aquí tiene dos ventajas importantes: se evita que los usuarios vean un traceback, y se deja que el programa siga analizando los textos que sí encuentra. Si no se capturara el `FileNotFoundError`, el usuario vería un traceback completo y el programa se detendría después de intentar analizar *Siddhartha*: nunca analizaría *Moby Dick* ni *Little Women*.

### Failing Silently

En el ejemplo anterior se informó a los usuarios que uno de los archivos no estaba. Pero **no hace falta informar cada excepción** que se captura. A veces se quiere que el programa falle en silencio y continúe como si nada. Se escribe el `try` como siempre, pero se le dice explícitamente a Python que no haga nada en el `except`, con la instrucción `pass`:

In [15]:
from pathlib import Path

def count_words(path):
    """Count the approximate number of words in a file."""
    try:
        contents = path.read_text(encoding='utf-8')
    except FileNotFoundError:
        pass
    else:
        # Count the approximate number of words in the file:
        words = contents.split()
        num_words = len(words)
        print(f"The file {path} has about {num_words} words.")

filenames = ['alice.txt', 'siddhartha.txt', 'moby_dick.txt',
        'little_women.txt']
for filename in filenames:
    path = Path(filename)
    count_words(path)

The file alice.txt has about 29594 words.
The file moby_dick.txt has about 215864 words.
The file little_women.txt has about 189142 words.


Ahora, cuando se produce un `FileNotFoundError`, se ejecuta el código del `except`, pero no pasa nada: no hay traceback ni mensaje. Los usuarios ven las cuentas de los archivos que existen, pero ninguna indicación de que uno faltó.
`pass` también sirve como **marcador de posición** (*placeholder*): un recordatorio de que se eligió no hacer nada en un punto del programa, y que quizás se quiera hacer algo ahí más adelante. Por ejemplo, se podrían escribir los nombres de archivos que faltan en un archivo `missing_files.txt` que los usuarios no ven, pero que sirve para revisar después.

### Deciding Which Errors to Report

¿Cómo saber cuándo informar un error al usuario y cuándo dejar que el programa falle en silencio? Si los usuarios saben qué textos se deben analizar, quizás agradezcan un mensaje que explique por qué algunos no se analizaron. Si esperan ver resultados pero no saben qué libros se deben analizar, quizás no necesiten saber que algunos no estaban. Dar información que los usuarios no buscan puede disminuir la usabilidad del programa. Las estructuras de manejo de errores de Python permiten controlar con detalle cuánto compartir con los usuarios cuando algo sale mal; es decisión del programador cuánta información compartir.
El código bien escrito y probado no es muy propenso a errores internos, como errores de sintaxis o lógicos. Pero cada vez que el programa depende de algo externo (la entrada del usuario, la existencia de un archivo, la disponibilidad de una conexión de red), existe la posibilidad de que se produzca una excepción. Con algo de experiencia se aprende dónde incluir bloques de manejo de excepciones y cuánto informar a los usuarios de los errores.

### Comparison: Exceptions

| | `ZeroDivisionError` | `FileNotFoundError` | `ValueError` |
|---|---|---|---|
| Cuándo ocurre | se divide por cero | el archivo no existe en esa ruta | se convierte un texto no numérico con `int()` |
| Ejemplo | `5/0` | `Path('alice.txt').read_text()` sin el archivo | `int('abc')` |
| Mensaje amable (ejemplo del libro) | `You can't divide by 0!` | `Sorry, the file alice.txt does not exist.` | (ejercicio 10-6) |

| | `try` | `except` | `else` |
|---|---|---|---|
| Qué contiene | **solo** el código que podría fallar | qué hacer si ocurre esa excepción | código que depende de que el `try` haya funcionado |
| Se ejecuta | siempre | solo si hubo la excepción indicada | solo si **no** hubo excepción |
| Ejemplo | `answer = int(first_number) / int(second_number)` | `print("You can't divide by 0!")` | `print(answer)` |

| | Informar el error | Fallar en silencio |
|---|---|---|
| Bloque `except` | `print(f"Sorry, the file {path} does not exist.")` | `pass` |
| El usuario ve | el mensaje | nada |
| Cuándo usarlo | los usuarios saben qué archivos esperan | los usuarios no necesitan saber que un archivo faltó |

| | Sin `try-except` | Con `try-except` |
|---|---|---|
| Si un archivo falta | traceback y el programa se detiene | mensaje (o nada) y el programa sigue con los demás |
| Qué ve el usuario | un traceback confuso | un mensaje claro |

### Try it Yourself

> 10-6. Addition:
One common problem when prompting for numerical input occurs when people provide text instead of numbers. When you try to convert the input to an int, you’ll get a ValueError. Write a program that prompts for two numbers. Add them together and print the result. Catch the ValueError if either input value is not a number, and print a friendly error message. Test your program by entering two numbers and then by entering some text instead of a number.

> 10-7. Addition Calculator:
Wrap your code from Exercise 10-5 in a while loop so the user can continue entering numbers, even if they make a mistake and enter text instead of a number.

> 10-8. Cats and Dogs:
Make two files, cats.txt and dogs.txt. Store at least three names of cats in the first file and three names of dogs in the second file. Write a program that tries to read these files and print the contents of the file to the screen. Wrap your code in a try-except block to catch the FileNotFound error, and print a friendly message if a file is missing. Move one of the files to a different location on your system, and make sure the code in the except block executes properly.

> 10-9. Silent Cats and Dogs:
Modify your except block in Exercise 10-7 to fail silently if either file is missing.

> 10-10. Common Words:
Visit Project Gutenberg (https://gutenberg.org) and find a few texts you’d like to analyze. Download the text files for these works, or copy the raw text from your browser into a text file on your computer.
You can use the count() method to find out how many times a word or phrase appears in a string. For example, the following code counts the number of times 'row' appears in a string:

In [16]:
line = "Row, row, row your boat"
print(line.count('row'))
print(line.lower().count('row'))

2
3


Notice that converting the string to lowercase using lower() catches all appearances of the word you’re looking for, regardless of how it’s formatted.
Write a program that reads the files you found at Project Gutenberg and determines how many times the word 'the' appears in each text. This will be an approximation because it will also count words such as 'then' and 'there'. Try counting 'the ', with a space in the string, and see how much lower your count is.

In [ ]:
# 10-6

In [ ]:
# 10-7

In [ ]:
# 10-8

In [ ]:
# 10-9

In [ ]:
# 10-10

## Storing Data

Muchos programas piden información al usuario: preferencias en un juego, datos para una visualización... Esa información se guarda en estructuras como listas y diccionarios, y cuando el usuario cierra el programa casi siempre se quiere guardar lo que ingresó. Una forma simple es usar el módulo **`json`**.
`json` convierte estructuras de datos simples de Python en strings con formato JSON, y luego se cargan los datos desde el archivo la próxima vez que corre el programa. También sirve para compartir datos entre programas de Python distintos. Y como JSON no es específico de Python, los datos se pueden compartir con personas que trabajan en muchos otros lenguajes. Es un formato útil y portable, y fácil de aprender.

> El formato JSON (JavaScript Object Notation) se desarrolló originalmente para JavaScript, pero hoy lo usan muchos lenguajes, incluido Python.

### Using json.dumps() and json.loads()

Un programa que guarda un conjunto de números con `json.dumps()`, y otro que los lee de vuelta con `json.loads()`.
`json.dumps()` recibe un dato y lo convierte a formato JSON. Devuelve un **string**, que se puede escribir en un archivo:

In [17]:
from pathlib import Path
import json

numbers = [2, 3, 5, 7, 11, 13]

path = Path('numbers.json')
contents = json.dumps(numbers)
path.write_text(contents)

Se importa `json`, se crea una lista de números y se elige un nombre de archivo (se acostumbra usar la extensión `.json` para indicar que los datos tienen formato JSON). `json.dumps()` genera un string con la representación JSON del dato, y se escribe con `write_text()`. El programa no muestra nada, pero el archivo `numbers.json` guarda los datos en un formato que se parece a Python: `[2, 3, 5, 7, 11, 13]`.

Un programa aparte lee la lista con `json.loads()`:

In [18]:
from pathlib import Path
import json

path = Path('numbers.json')
contents = path.read_text()
numbers = json.loads(contents)

print(numbers)

[2, 3, 5, 7, 11, 13]


Como el archivo de datos es solo un archivo de texto con cierto formato, se puede leer con `read_text()`. Después se pasa el contenido a `json.loads()`, que recibe un string con formato JSON y devuelve un objeto de Python (aquí, una lista) que se asigna a `numbers`. Se imprime la lista recuperada y es la misma que se creó en `number_writer.py`. Es una forma simple de compartir datos entre dos programas.

### Saving and Reading User-Generated Data

Guardar datos con `json` es útil con datos generados por el usuario, porque si no se guarda su información de alguna forma, se pierde cuando el programa termina. Ejemplo: se pide el nombre del usuario la primera vez que corre el programa y se recuerda la próxima vez. Primero, guardar el nombre:

In [ ]:
from pathlib import Path
import json

username = input("What is your name? ")

path = Path('username.json')
contents = json.dumps(username)
path.write_text(contents)

print(f"We'll remember you when you come back, {username}!")

Un programa nuevo saluda a un usuario cuyo nombre ya se guardó:

In [ ]:
from pathlib import Path
import json

path = Path('username.json')
contents = path.read_text()
username = json.loads(contents)

print(f"Welcome back, {username}!")

Hay que combinar ambos programas en un solo archivo. Cuando alguien ejecute `remember_me.py`, se quiere recuperar su nombre de usuario si es posible; si no, se pide y se guarda en `username.json` para la próxima vez. Se podría usar un `try-except` para responder si `username.json` no existe, pero en vez de eso se usa un método útil de `pathlib`:

In [ ]:
from pathlib import Path
import json

path = Path('username.json')
if path.exists():
    contents = path.read_text()
    username = json.loads(contents)
    print(f"Welcome back, {username}!")
else:
    username = input("What is your name? ")
    contents = json.dumps(username)
    path.write_text(contents)
    print(f"We'll remember you when you come back, {username}!")

Los objetos `Path` tienen muchos métodos útiles. `exists()` devuelve `True` si un archivo o carpeta existe y `False` si no. Si `username.json` existe, se carga el nombre y se saluda; si no existe, se pide un nombre, se guarda y se avisa que se recordará.
Con cualquiera de los dos bloques el resultado es un nombre de usuario y un saludo adecuado. La primera vez que se ejecuta, la salida es `What is your name?` + `We'll remember you when you come back, Eric!`; las veces siguientes, `Welcome back, Eric!`.
Aunque aquí el dato es un solo string, el programa funcionaría igual con cualquier dato que se pueda convertir a un string con formato JSON.

### Refactoring

A menudo se llega a un punto en que el código funciona, pero se podría mejorar dividiéndolo en una serie de funciones que tengan trabajos específicos. Este proceso se llama **refactoring**, y hace el código más limpio, más fácil de entender y de ampliar.
Para refactorizar `remember_me.py` se mueve la mayor parte de su lógica a una o más funciones. Como el foco es saludar al usuario, se mueve todo el código a una función `greet_user()`:

In [ ]:
from pathlib import Path
import json

def greet_user():
    """Greet the user by name."""
    path = Path('username.json')
    if path.exists():
        contents = path.read_text()
        username = json.loads(contents)
        print(f"Welcome back, {username}!")
    else:
        username = input("What is your name? ")
        contents = json.dumps(username)
        path.write_text(contents)
        print(f"We'll remember you when you come back, {username}!")

greet_user()

Al usar una función, los comentarios se reescriben como un docstring. Pero `greet_user()` hace más que saludar: también recupera un nombre guardado y pide uno nuevo si no existe. Se refactoriza para que haga menos tareas. Primero, el código que recupera el nombre guardado pasa a otra función:

In [ ]:
from pathlib import Path
import json

def get_stored_username(path):
    """Get stored username if available."""
    if path.exists():
        contents = path.read_text()
        username = json.loads(contents)
        return username
    else:
        return None

def greet_user():
    """Greet the user by name."""
    path = Path('username.json')
    username = get_stored_username(path)
    if username:
        print(f"Welcome back, {username}!")
    else:
        username = input("What is your name? ")
        contents = json.dumps(username)
        path.write_text(contents)
        print(f"We'll remember you when you come back, {username}!")

greet_user()

`get_stored_username()` tiene un propósito claro (lo dice su docstring): recupera el nombre guardado y lo devuelve si lo encuentra. Si la ruta que se le pasa no existe, devuelve `None`. Es una buena práctica: **una función debería devolver el valor esperado o `None`**. Así se puede hacer una prueba simple con el valor de retorno: si se recuperó un nombre, se saluda; si no, se pide uno nuevo.
Hay que sacar un bloque más de `greet_user()`: si el nombre no existe, el código que pide uno nuevo pasa a una función dedicada a eso:

In [ ]:
from pathlib import Path
import json

def get_stored_username(path):
    """Get stored username if available."""
    if path.exists():
        contents = path.read_text()
        username = json.loads(contents)
        return username
    else:
        return None

def get_new_username(path):
    """Prompt for a new username."""
    username = input("What is your name? ")
    contents = json.dumps(username)
    path.write_text(contents)
    return username

def greet_user():
    """Greet the user by name."""
    path = Path('username.json')
    username = get_stored_username(path)
    if username:
        print(f"Welcome back, {username}!")
    else:
        username = get_new_username(path)
        print(f"We'll remember you when you come back, {username}!")

greet_user()

Cada función de esta versión final tiene un único propósito claro. Se llama a `greet_user()`, que imprime el mensaje apropiado: da la bienvenida a un usuario existente o saluda a uno nuevo. Para eso llama a `get_stored_username()`, que solo recupera el nombre guardado si existe, y si hace falta, a `get_new_username()`, que solo obtiene y guarda un nombre nuevo. Esta **compartimentación** del trabajo es esencial para escribir código claro que sea fácil de mantener y ampliar.

### Comparison: Storing Data

| | `json.dumps(dato)` | `json.loads(texto)` |
|---|---|---|
| Dirección | dato de Python → texto JSON | texto JSON → dato de Python |
| Recibe | una lista, un diccionario, un string, un número... | un string con formato JSON |
| Devuelve | un string | un objeto de Python |
| Se combina con | `path.write_text()` | `path.read_text()` |
| Ejemplo | `json.dumps([2, 3, 5])` → `'[2, 3, 5]'` | `json.loads('[2, 3, 5]')` → `[2, 3, 5]` |

| | `path.exists()` | `try-except FileNotFoundError` |
|---|---|---|
| Cómo detecta el archivo faltante | pregunta antes si existe | intenta leerlo y captura el error |
| Se usó en | `remember_me.py` | `alice.py`, `word_count.py` |
| Cuándo usarlo | un archivo puede o no existir y ambas situaciones son normales | un archivo debería existir y su ausencia es un problema |

| Versión de `remember_me.py` | Qué hace cada parte |
|---|---|
| Primera | todo en el archivo, sin funciones |
| `greet_user()` | todo el código dentro de una función |
| `get_stored_username()` | separa la **recuperación** del nombre (devuelve el nombre o `None`) |
| `get_new_username()` | separa la **solicitud y el guardado** del nombre |
| Final | `greet_user()` solo decide qué mensaje mostrar |

### Try it Yourself

> 10-11. Favorite Number:
Write a program that prompts for the user’s favorite number. Use json.dumps() to store this number in a file. Write a separate program that reads in this value and prints the message “I know your favorite number! It’s _____.”

> 10-12. Favorite Number Remembered:
Combine the two programs you wrote in Exercise 10-11 into one file. If the number is already stored, report the favorite number to the user. If not, prompt for the user’s favorite number and store it in a file. Run the program twice to see that it works.

> 10-13. User Dictionary:
The remember_me.py example only stores one piece of information, the username. Expand this example by asking for two more pieces of information about the user, then store all the information you collect in a dictionary. Write this dictionary to a file using json.dumps(), and read it back in using json.loads(). Print a summary showing exactly what your program remembers about the user.

> 10-14. Verify User:
The final listing for remember_me.py assumes either that the user has already entered their username or that the program is running for the first time. We should modify it in case the current user is not the person who last used the program.
Before printing a welcome back message in greet_user(), ask the user if this is the correct username. If it’s not, call get_new_username() to get the correct username.

In [ ]:
# 10-11

In [ ]:
# 10-12

In [ ]:
# 10-13

In [ ]:
# 10-14

## Comparison: Chapter 10

| Si necesito... | Uso |
|---|---|
| Leer todo un archivo de texto | `Path('archivo.txt').read_text()` |
| Recorrer un archivo línea por línea | `contents.splitlines()` + `for` |
| Quitar el espacio sobrante | `.rstrip()` / `.lstrip()` / `.strip()` |
| Escribir en un archivo | `path.write_text(texto)` |
| Escribir varias líneas | armar un string con `+=` y `\n`, y llamar una sola vez a `write_text()` |
| Contar las palabras de un texto | `len(contents.split())` |
| Evitar que el programa se caiga por un error | `try-except` |
| Poner el código que depende del `try` | `else` |
| Ignorar un error sin avisar | `except TipoDeError: pass` |
| Saber si un archivo existe | `path.exists()` |
| Guardar datos que sobrevivan al programa | `json.dumps()` + `write_text()` |
| Recuperar datos guardados | `read_text()` + `json.loads()` |
| Que una función avise que no encontró algo | que devuelva `None` |
| Hacer el código más claro | refactoring en funciones con un solo propósito |

| | Leer | Escribir | Guardar datos | Manejar errores |
|---|---|---|---|---|
| Herramienta | `read_text()` | `write_text()` | `json.dumps()` / `json.loads()` | `try-except-else` |
| Módulo | `pathlib` | `pathlib` | `json` | (incluido en Python) |
| Trabaja con | strings | strings | listas, diccionarios, strings... | excepciones |